In [ ]:
# Run this cell to import pyspark and to define start_spark() and stop_spark()

import findspark

findspark.init()

import getpass
import pyspark
import random
import re

from IPython.display import display, HTML
from pyspark import SparkContext
from pyspark.sql import SparkSession


# Constants used to interact with Azure Blob Storage using the hdfs command or Spark

global username

username = re.sub('@.*', '', getpass.getuser())


# Functions used below

def dict_to_html(d):
    """Convert a Python dictionary into a two column table for display.
    """

    html = []

    html.append(f'<table width="100%" style="width:100%; font-family: monospace;">')
    for k, v in d.items():
        html.append(f'<tr><td style="text-align:left;">{k}</td><td>{v}</td></tr>')
    html.append(f'</table>')

    return ''.join(html)


def show_as_html(df, n=20):
    """Leverage existing pandas jupyter integration to show a spark dataframe as html.
    
    Args:
        n (int): number of rows to show (default: 20)
    """

    display(df.limit(n).toPandas())

    
def display_spark():
    """Display the status of the active Spark session if one is currently running.
    """
    
    if 'spark' in globals() and 'sc' in globals():

        name = sc.getConf().get("spark.app.name")

        html = [
            f'<p><b>Spark</b></p>',
            f'<p>The spark session is <b><span style="color:green">active</span></b>, look for <code>{name}</code> under the running applications section in the Spark UI.</p>',
            f'<ul>',
            f'<li><a href="http://localhost:{sc.uiWebUrl.split(":")[-1]}" target="_blank">Spark Application UI</a></li>',
            f'</ul>',
            f'<p><b>Config</b></p>',
            dict_to_html({k: v for k, v in sc.getConf().getAll() if not re.search(r"(secret|password|token|credential|sas|account\.key)", k, re.I)}),
            f'<p><b>Notes</b></p>',
            f'<ul>',
            f'<li>The spark session <code>spark</code> and spark context <code>sc</code> global variables have been defined by <code>start_spark()</code>.</li>',
            f'<li>Please run <code>stop_spark()</code> before closing the notebook or restarting the kernel or kill <code>{name}</code> by hand using the link in the Spark UI.</li>',
            f'</ul>',
        ]
        display(HTML(''.join(html)))
        
    else:
        
        html = [
            f'<p><b>Spark</b></p>',
            f'<p>The spark session is <b><span style="color:red">stopped</span></b>, confirm that <code>{username} (notebook)</code> is under the completed applications section in the Spark UI.</p>',
            f'<ul>',
            f'<li><a href="http://mathmadslinux2p.canterbury.ac.nz:8080/" target="_blank">Spark UI</a></li>',
            f'</ul>',
        ]
        display(HTML(''.join(html)))


# Functions to start and stop spark

def start_spark(executor_instances=2, executor_cores=1, worker_memory=1, master_memory=1):
    """Start a new Spark session and define globals for SparkSession (spark) and SparkContext (sc).
    
    Args:
        executor_instances (int): number of executors (default: 2)
        executor_cores (int): number of cores per executor (default: 1)
        worker_memory (float): worker memory (default: 1)
        master_memory (float): master memory (default: 1)
    """

    global spark
    global sc

    cores = executor_instances * executor_cores
    partitions = cores * 4
    port = 4000 + random.randint(1, 999)

    spark = (
        SparkSession.builder
        .config("spark.driver.extraJavaOptions", f"-Dderby.system.home=/tmp/{username}/spark/")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.executor.instances", str(executor_instances))
        .config("spark.executor.cores", str(executor_cores))
        .config("spark.cores.max", str(cores))
        .config("spark.driver.memory", f'{master_memory}g')
        .config("spark.executor.memory", f'{worker_memory}g')
        .config("spark.driver.maxResultSize", "0")
        .config("spark.sql.shuffle.partitions", str(partitions))
        .config("spark.kubernetes.container.image", "madsregistry001.azurecr.io/hadoop-spark:v3.3.5-openjdk-8")
        .config("spark.kubernetes.container.image.pullPolicy", "IfNotPresent")
        .config("spark.kubernetes.memoryOverheadFactor", "0.3")
        .config("spark.memory.fraction", "0.1")
        .config("spark.app.name", f"{username} (notebook)")
        .getOrCreate()
    )
    sc = SparkContext.getOrCreate()
    
    display_spark()

    
def stop_spark():
    """Stop the active Spark session and delete globals for SparkSession (spark) and SparkContext (sc).
    """

    global spark
    global sc

    if 'spark' in globals() and 'sc' in globals():

        spark.stop()

        del spark
        del sc

    display_spark()


# Make css changes to improve spark output readability

html = [
    '<style>',
    'pre { white-space: pre !important; }',
    'table.dataframe td { white-space: nowrap !important; }',
    'table.dataframe thead th:first-child, table.dataframe tbody th { display: none; }',
    '</style>',
]
display(HTML(''.join(html)))

In [ ]:
# Run this cell to start a spark session in this notebook

start_spark(executor_instances=4, executor_cores=4, worker_memory=4, master_memory=4)

In [ ]:
# We need to import pyplot from matplotlib in order to visualize our data locally 

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

from pyspark.sql import Row, DataFrame, Window, functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import when, col

from pyspark.ml.functions import vector_to_array
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.sql.types import NumericType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer
from pyspark.ml.functions import vector_to_array

import numpy as np

In [ ]:
# Other imports to be used locally

import datetime
import numpy as np

np.set_printoptions(edgeitems=5, threshold=100, precision=4)

In [ ]:
# Helper functions

def show_class_balance(data, name="data", labelCol="label"):
    """Helper function to show class balance based on label.
    
    Note that this function does not return anything.

    Args:
        data (pyspark.sql.DataFrame): datafame with label
        name (str): name to print above metrics for readability 
        labelCol (str): label column name
    """

    total = data.count()
    counts = data.groupBy(labelCol).count().toPandas()
    counts["ratio"] = counts["count"] / total

    print(f'Class balance [{name}]')
    print(f'')
    print(f'total:   {total}')
    print(f'counts:')
    print(counts)
    print(f'')

    
def with_custom_prediction(
    pred,
    threshold,
    probabilityCol="probability",
    customPredictionCol="customPrediction",
):
    """Helper function to select a custom prediction column for a custom classification threshold.
    
    Args:
        pred (pyspark.sql.DataFrame): datafame with column for probability 
        threshold (float): classification threshold
        probabilityCol (str): probability column name
        customPredictionCol (str): new custom prediction column name
    
    Returns:
        pred (pyspark.sql.DataFrame): dataframe with new colum for custom prediction
    """

    classification_udf = F.udf(lambda x: int(x[1] > threshold), IntegerType())
    
    return pred.withColumn(customPredictionCol, classification_udf(F.col(probabilityCol)))


def show_metrics(
    pred,
    name="data",
    threshold=0.5,
    labelCol="label",
    predictionCol="prediction",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
):
    """Helper function to evaluate and show metrics based on a custom classification threshold.
    
    Note that this function does not return anything.
    
    Args:
        pred (pyspark.sql.DataFrame): datafame with column for probability 
        name (str): name to print above metrics for readability 
        threshold (float): classification threshold (default: 0.5)
        predictionCol (str): prediction column name
        rawPredictionCol (str): raw prediction column name
        probabilityCol (str): probability column name
    """

    if threshold != 0.5:

        predictionCol = "customPrediction"
        pred = with_custom_prediction(pred, threshold, probabilityCol=probabilityCol, customPredictionCol=predictionCol)

    total = pred.count()

    nP_actual = pred.filter((F.col(labelCol) == 1)).count()
    nN_actual = pred.filter((F.col(labelCol) == 0)).count()

    nP = pred.filter((F.col(predictionCol) == 1)).count()
    nN = pred.filter((F.col(predictionCol) == 0)).count()
    TP = pred.filter((F.col(predictionCol) == 1) & (F.col(labelCol) == 1)).count()
    FP = pred.filter((F.col(predictionCol) == 1) & (F.col(labelCol) == 0)).count()
    FN = pred.filter((F.col(predictionCol) == 0) & (F.col(labelCol) == 1)).count()
    TN = pred.filter((F.col(predictionCol) == 0) & (F.col(labelCol) == 0)).count()

    if TP + FP > 0:
        precision = TP / (TP + FP)
    else:
        precision = 0
        
    recall = TP / (TP + FN)
    accuracy = (TP + TN) / total

    binary_evaluator = BinaryClassificationEvaluator(
        rawPredictionCol=rawPredictionCol,
        labelCol=labelCol,
        metricName='areaUnderROC',
    )
    auroc = binary_evaluator.evaluate(pred)

    print(f'Metrics [{name}]')
    print(f'')
    print(f'threshold: {threshold}')
    print(f'')
    print(f'total:     {total}')
    print(f'')
    print(f'nP actual: {nP_actual}')
    print(f'nN actual: {nN_actual}')
    print(f'')
    print(f'nP:        {nP}')
    print(f'nN:        {nN}')
    print(f'')
    print(f'TP         {TP}')
    print(f'FP         {FP}')
    print(f'FN         {FN}')
    print(f'TN         {TN}')
    print(f'')
    print(f'precision: {precision:.8f}')
    print(f'recall:    {recall:.8f}')
    print(f'accuracy:  {accuracy:.8f}')
    print(f'')
    print(f'auroc:     {auroc:.8f}')


def expand(x, s=0.05, d=0):
    """Expand a two element array about its center point by a relative scale or a fixed offset.
    Args:
        x (list|np.array): two element array
        s (float): relative scale to expand array based on its width x[1] - x[0]
        d (float): fixed offset to expand array
    Returns:
        x (np.array): expanded two element array
    """
    
    x = np.array(x)
    d = d + s * (x[1] - x[0])
    
    return x + np.array([-d, d])

In [ ]:
# Determine ideal number of partitions

conf = sc.getConf()

N = int(conf.get("spark.executor.instances"))
M = int(conf.get("spark.executor.cores"))
partitions = 4 * N * M

print(f'ideal # partitions = {partitions}')

## Q1 Audio Feature Analysis

In [ ]:
# Mapping from attribute type string to Spark DataType
type_mapping = {
    "string": StringType(),
    "numeric": DoubleType(),
    "real": DoubleType(),
    "float": DoubleType(),
}

def get_schema_from_attributes(prefix):
    """
    Read the attributes file for `prefix` and build a StructType
    for the matching features file.
    """
    attr_path = f"wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/audio/attributes/{prefix}.attributes.csv"
    # Load attribute names and types
    attr_df = spark.read.csv(attr_path, header=False, inferSchema=True)

    attrs = attr_df.collect()

    fields = []
    # Map each attribute row to a StructField
    for row in attrs:
        name = row[0]
        dtype_key = row[1].lower()
        spark_type = type_mapping.get(dtype_key, StringType())
        fields.append(StructField(name, spark_type, True))

    return StructType(fields)

# List all 4 attribute prefixes
attribute_prefixes = [
    "msd-jmir-area-of-moments-all-v1.0",
    "msd-jmir-lpc-all-v1.0",
    "msd-jmir-spectral-all-all-v1.0",
    "msd-marsyas-timbral-v1.0"
]

# Generate and save schema for each attribute file
schema_map = {}
for prefix in attribute_prefixes:
    schema_map[prefix] = get_schema_from_attributes(prefix)

#### Rename the Schema

In [ ]:

# Abbreviation mapping for key terms
abbr = {
    "mean":              "mean",
    "standard_deviation": "std",
    "average":            "avg",
    "overall":            "ovr",
    "component":            "comp",
    "spectral_centroid":            "cent",
    "spectral":            "spec",
    "zero_crossings":            "zcr",
    "area_method_of_moments":            "amom",
    "method_of_moments":            "mom",
    "moments":            "mon",
    "root_mean_square":            "rms",
    "spectral_variability":            "variab",
}

def rename(col_name: str, idx: int, prefix_short: str) -> str:
    """
    1. Lowercase & replace non-alphanumerics with underscores.
    2. Substitute full words with abbreviations.
    3. Split into tokens, then:
       - Keep each abbr token wrapped in underscores.
       - Group other tokens: take first letters, merge into one string, wrap in underscores.
    4. Remove duplicate segments, prepend prefix_short, and append an index suffix.
    5. Enforce max length of 30 chars (including suffix).
    """
    # 1. normalize
    s = re.sub(r'[^A-Za-z0-9]+', '_', col_name.lower()).strip('_')
    # 2. replace with abbreviations
    for full, short in abbr.items():
        s = s.replace(full, short)
    tokens = s.split('_')

    parts, i = [], 0
    abbr_vals = set(abbr.values())
    # 3. process tokens
    while i < len(tokens):
        t = tokens[i]
        if t in abbr_vals:
            parts.append(f"_{t}_")
            i += 1
        else:
            # collect a run of non-abbr tokens
            letters = []
            while i < len(tokens) and tokens[i] not in abbr_vals:
                letters.append(tokens[i][0])
                i += 1
            parts.append(f"_{''.join(letters)}_")

    # remove duplicates while preserving order
    seen = set()
    uniq = []
    for seg in parts:
        if seg not in seen:
            uniq.append(seg)
            seen.add(seg)

    core = "".join(uniq)
    suffix = f"_{idx}"
    max_len = 30
    base = (prefix_short + core)[: max_len - len(suffix)]
    return f"{base}{suffix}"

# Process all datasets
dfs = {}
for prefix in attribute_prefixes:
    # derive short prefix
    p_short = "_".join(prefix.split("-")[1:3])
    schema = schema_map[prefix]
    path = (
        "wasbs://campus-data@madsstorage002.blob.core.windows.net"
        f"/msd/audio/features/{prefix}.csv"
    )

    # load without header
    df = spark.read.csv(path, schema=schema, header=False)

    # rename columns with index
    new_names = [
        rename(col, idx, p_short)
        for idx, col in enumerate(df.columns, start=1)
    ]
    df = df.toDF(*new_names)

    # store & inspect
    dfs[prefix] = df
    # df.printSchema()
    # df.show(3)


#### Rename the track_id for joining and drop the empty columns

In [ ]:
for prefix, df in dfs.items():
    # 1. rename last column to "track_id"
    df = df.withColumnRenamed(df.columns[-1], "track_id")
    # 2. remove any single-quote chars from track_id
    df = df.withColumn(
        "track_id",
        F.regexp_replace(F.col("track_id"), "'", "")
    )
    # drop columns where all values are zero
    zero_cols = [c for c in df.columns if c != "track_id" and df.filter(F.col(c) != 0).count() == 0]
    if zero_cols:
        df = df.drop(*zero_cols)
    # 3. register as temp view 
    view_name = prefix.replace('-', '_').replace('.', '_')
    df.createOrReplaceTempView(view_name)
    # 4. update dfs in place
    dfs[prefix] = df

#### Join dataframes

In [ ]:
# Collect all feature DataFrames into a list
df_lists = list(dfs.values())

# Start with the first DataFrame
joined_features = df_lists[0]

# Iteratively full outer join with each remaining DataFrame on track_id
for df in df_lists[1:]:
    joined_features = joined_features.join(df, on="track_id", how="inner")

joined_features = joined_features.repartition(partitions, "track_id")
joined_features.cache()

joined_features.printSchema()

#### Save the statistics to excel

In [ ]:
df_summary = joined_features.describe()  

output_path_parquet = "joined_feature_describe.parquet"
df_summary.write.mode("overwrite").parquet(output_path_parquet)
print(f"Description saved to {output_path_parquet}")

# view the summary
df_summary.show()

In [ ]:
joined_features.count()

In [ ]:
# Identify numeric feature columns except 'track_id'
feature_cols = [
    f.name for f in joined_features.schema.fields
    if f.name != "track_id"
    and joined_features.schema[f.name].dataType.typeName() in ['double', 'float', 'int', 'long']
]

cols = feature_cols[:4]  

pdf = joined_features.select(cols).toPandas()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), cols):
    ax.boxplot(pdf[col].dropna())
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
#  Assemble features into a single vector, skipping rows with nulls
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)
assembled_df = assembler.transform(joined_features)

#  Print schema and show sample records
assembled_df.printSchema()
show_as_html(assembled_df)
print(f"Number of columns: {len(assembled_df.columns)}")

#  Compute descriptive statistics for each numeric feature
stats_df = assembled_df.select(feature_cols).describe()

#    Convert to pandas DataFrame and set 'summary' as column index
stats_pd = stats_df.toPandas().set_index("summary").T

#  Select the assembled feature vector for correlation calculation
vector_df = assembled_df.select("features")

#  Compute Pearson correlation matrix (column names remain as original feature_cols)
corr_matrix = (
    Correlation
      .corr(vector_df, "features", "pearson")
      .head()[0]
      .toArray()
)

#  Convert correlation matrix to pandas DataFrame with feature names as index and columns
corr_df = pd.DataFrame(corr_matrix, index=feature_cols, columns=feature_cols)


In [ ]:
# draw heatmap
plt.figure(figsize=(20, 20))
plt.imshow(corr_df, aspect='auto')
plt.colorbar()
plt.xticks(
    range(len(corr_df.columns)),
    corr_df.columns,
    rotation=90,
    fontsize=6
)
plt.yticks(
    range(len(corr_df.index)),
    corr_df.index,
    fontsize=6
)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()

# save the figure to a file
plt.savefig("feature_corr_heatmap.png", dpi=300) 

plt.show()

In [ ]:
#  Identify highly correlated features (absolute correlation > 0.9)
threshold = 0.9
upper = np.triu(np.ones(corr_df.shape), k=1).astype(bool)  # upper‐triangle mask
high_corr = corr_df.where(upper).abs()                    # ignore diagonal and duplicates
to_drop = [col for col in corr_df.columns if high_corr[col].max() > threshold]

#  Drop the highly correlated features from assembled_df
filtered_df = assembled_df.drop(*to_drop)

#  Extract each element of the "features" vector into its own column,
remaining_features = [c for c in feature_cols if c not in to_drop]

for i, fname in enumerate(feature_cols):
    if fname in remaining_features:
        # use vector_to_array to expand "features" into an array, then pick index i
        filtered_df = filtered_df.withColumn(
            fname,
            vector_to_array("features")[i]
        )

#  Select only "track_id" plus the newly expanded columns for remaining features
reduced_features = filtered_df.select(["track_id"] + remaining_features)


#  Print out which features were removed
print("Dropped features:", to_drop)

In [ ]:
show_as_html(reduced_features)

In [ ]:
len(reduced_features.columns)

In [ ]:
# reduced_features.printSchema()

In [ ]:
show_as_html(reduced_features)

In [ ]:
# 1. Select the numeric columns to process (exclude the "track_id" column)
numeric_cols = [c for c in reduced_features.columns if c != "track_id"]

# 2. Define a VectorAssembler to combine all numeric columns into a single feature vector
assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="features",
    handleInvalid="skip"
)

# 3. Apply the assembler to the DataFrame to produce the "features" column
assembled_df = assembler.transform(reduced_features)

# 4. Select only the assembled feature vector column
vector_df = assembled_df.select("features")


# 5. calculate Pearson correlation
corr_matrix = Correlation.corr(vector_df, "Features", "pearson") \
                        .head()[0] \
                        .toArray()

# 6. convert to pandas DataFrame
corr_df = pd.DataFrame(corr_matrix, index=numeric_cols, columns=numeric_cols)

# Plot heatmap with a large figure and small fonts
plt.figure(figsize=(20, 20))
plt.imshow(corr_df, aspect='auto')
plt.colorbar()
plt.xticks(
    np.arange(len(numeric_cols)),
    numeric_cols,
    rotation=90,
    fontsize=6
)
plt.yticks(
    np.arange(len(numeric_cols)),
    numeric_cols,
    fontsize=6
)
plt.title("Reduced Features Correlation Heatmap")
plt.tight_layout()

# save the figure to a file
plt.savefig("Reduced_feature_corr_heatmap.png", dpi=300) 

plt.show()

In [ ]:
# threshold2 = 0.9

# # Build an upper‐triangle mask to ignore self‐correlation and duplicates
# upper2 = np.triu(np.ones(corr_df.shape), k=1).astype(bool)

# # Apply mask and take absolute values
# high_corr2 = corr_df.where(upper2).abs()

# # Identify features to drop in this second round
# to_drop2 = [col for col in corr_df.columns if high_corr2[col].max() > threshold2]

# # Drop these highly correlated features from reduced_features
# final_df = reduced_features.drop(*to_drop2)

# # Print out which features were removed in the second pass
# print("Second round dropped features:", to_drop2)


#### b)

In [ ]:
# path to the genre file
path = (
    "wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/genre/msd-MAGD-genreAssignment.tsv"
)

# load the TSV file, no header, infer schema
genre_df = spark.read.csv(path, sep='\t', header=False, inferSchema=True)

# rename columns for clarity
genre_df = genre_df.withColumnRenamed("_c0", "track_id") \
           .withColumnRenamed("_c1", "genre")

# count how many tracks per genre
counts = genre_df.groupBy("genre") \
             .count() \
             .orderBy("count", ascending=False)

# show the top genres
counts.show(10)

# convert to Pandas for plotting
pdf = counts.toPandas()

# plot the genre distribution
plt.figure(figsize=(10, 6))
plt.bar(pdf['genre'], pdf['count'])
plt.xticks(rotation=45, ha='right')
plt.xlabel('Genre')
plt.ylabel('Number of Tracks')
plt.title('MAGD Genre Distribution')
plt.tight_layout()
plt.show()

#### c)

In [ ]:
#  Merge genre labels with your audio features DataFrame

feature_genre = reduced_features.join(genre_df, on="track_id", how="inner")

# 3. Verify the merged schema and preview
# feature_genre.printSchema()
feature_genre.show(5)
feature_genre.count()

## Q2

#### b)

In [ ]:
# # Add binary label: 1 if genre is Electronic, else 0
feature_genre = feature_genre.withColumn(
    "is_electronic",
    F.when(F.col("genre") == "Electronic", 1).otherwise(0)
)
# Show the distribution of the new binary label
feature_genre.groupBy("is_electronic").count().show()

In [ ]:
# Count the number of electronic vs other tracks
counts = feature_genre.groupBy("is_electronic").count().orderBy("is_electronic")

# Convert to Pandas for plotting
pdf_counts = counts.toPandas()

# Map label values to names
label_map = {0: "Other", 1: "Electronic"}
pdf_counts['label'] = pdf_counts['is_electronic'].map(label_map)

# Draw bar chart
plt.figure(figsize=(6, 4))
plt.bar(pdf_counts['label'], pdf_counts['count'])
plt.xlabel('Track Type')
plt.ylabel('Number of Tracks')
plt.title('Distribution of Electronic vs Other Tracks')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

#### c)

In [ ]:
# repartition by class
feature_genre = feature_genre.dropna()
feature_genre = feature_genre.repartition("is_electronic")

# filter electronic and other classes
electronic_df = feature_genre.filter(F.col("is_electronic") == 1)
other_df      = feature_genre.filter(F.col("is_electronic") == 0)

# stratified 80/20 split per class
elec_train, elec_test = electronic_df.randomSplit([0.8, 0.2], seed=42)
othr_train, othr_test = other_df.randomSplit([0.8, 0.2], seed=42)

# initial union
train_df = elec_train.union(othr_train)
test_df  = elec_test.union(othr_test)

# compute current counts
elec_count = train_df.filter(F.col("is_electronic") == 1).count()
othr_count = train_df.filter(F.col("is_electronic") == 0).count()

# desired minority count = one quarter of majority
desired_elec = othr_count / 4.0

if elec_count < desired_elec:
    # oversample minority to reach approx 1:4 ratio
    ratio = desired_elec / elec_count
    oversampled = elec_train.sample(withReplacement=True, fraction=ratio, seed=42)
    train_df = oversampled.union(othr_train)
elif elec_count > desired_elec:
    # undersample minority if it exceeds desired count
    ratio = desired_elec / elec_count
    undersampled = elec_train.sample(withReplacement=False, fraction=ratio, seed=42)
    train_df = undersampled.union(othr_train)

# verify new class counts
print("Train set class counts:")
train_df.groupBy("is_electronic").count().show()
print("Test set class counts:")
test_df.groupBy("is_electronic").count().show()


In [ ]:
# 1. Identify the numeric feature columns (exclude 'track_id' and the label 'is_electronic')
feature_cols = [
    c for c in train_df.columns
    if c not in ("track_id", "is_electronic", "genre")
       and isinstance(train_df.schema[c].dataType, NumericType)
]

# 2. Define a VectorAssembler to combine all selected features into a single vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
    # handleInvalid="skip"   # skip any rows that contain nulls
)

# 4. Build a Pipeline that applies the assembler 
pipeline = Pipeline(stages=[assembler])

# 5. Fit the pipeline on the training DataFrame to produce a PipelineModel
model = pipeline.fit(train_df)

# 6. Transform both training and test sets using the fitted PipelineModel, then cache
train_prepared = model.transform(train_df).cache()
test_prepared  = model.transform(test_df).cache()


In [ ]:
show_as_html(train_prepared)

In [ ]:
# Train a Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="is_electronic",
    standardization=True
)
lr_model = lr.fit(train_prepared)

# Apply the model to the test set
predictions = lr_model.transform(test_prepared).cache()

# Show class balance on the test set before prediction
show_class_balance(test_prepared, name="Test Set", labelCol="is_electronic")

# Evaluate and print metrics using the helper function
show_metrics(
    predictions,
    name="Logistic Regression on Test Set",
    labelCol="is_electronic",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
    predictionCol="prediction"
)

In [ ]:
# Train a Random Forest model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_electronic",
    numTrees=100,     # number of trees
    maxDepth=5,       # max depth per tree
    seed=42           # for reproducibility
)
rf_model = rf.fit(train_prepared)

# Apply the model to the test set
rf_predictions = rf_model.transform(test_prepared).cache()

# Show class balance on the test set before prediction
show_class_balance(test_prepared, name="Test Set", labelCol="is_electronic")

# Evaluate and print metrics using the helper function
show_metrics(
    rf_predictions,
    name="Random Forest on Test Set",
    labelCol="is_electronic",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
    predictionCol="prediction"
)

In [ ]:
from pyspark.ml.classification import GBTClassifier

# Train a GBTClassifier model
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="is_electronic",
    maxIter=50,      # number of boosting iterations
    maxDepth=5,      # maximum tree depth
    seed=42          # for reproducibility
)
gbt_model = gbt.fit(train_prepared)

# Apply the model to the test set
gbt_predictions = gbt_model.transform(test_prepared).cache()

# Show class balance on the test set before prediction
show_class_balance(test_prepared, name="Test Set", labelCol="is_electronic")

# Evaluate and print metrics using the helper function
show_metrics(
    gbt_predictions,
    name="GBTClassifier on Test Set",
    labelCol="is_electronic",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
    predictionCol="prediction"
)

## Q3 Predict across all genres

### b)

In [ ]:
feature_genre = feature_genre.drop("is_electronic")  # Remove the 'is_electronic' column

# 1. Convert 'genre' to an integer label 'genre_index'
indexer = StringIndexer(inputCol="genre", outputCol="genre_index")
index_model = indexer.fit(feature_genre)
df_indexed = index_model.transform(feature_genre)

# 2. Repartition by 'genre_index' and cache to speed up subsequent group aggregations
df_indexed = df_indexed.repartition("genre_index").cache()

# 3. Count the number of records for each genre
counts_df = (
    df_indexed
      .groupBy("genre", "genre_index")
      .count()
)

# 4. Compute the percentage share of each genre, rounded to two decimal places
total_count = df_indexed.count()
mutiple_df = (
    counts_df
      .withColumn("percentage", F.round(F.col("count") / F.lit(total_count) * 100, 2))
      .select("genre", "genre_index", "count", "percentage")
      .orderBy("genre_index")
)

# 5. Display the result
mutiple_df.show(truncate=False)


In [ ]:
# convert Spark DataFrame to Pandas DataFrame
mutiple_pdf = mutiple_df.toPandas()

# create a bar chart of count per genre
plt.figure()
mutiple_pdf.plot(kind='bar', x='genre', y='percentage', legend=False)
plt.xlabel('Genre')
plt.ylabel('Proportion')
plt.title('Proportion of Every Genre')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### c)

In [ ]:
# df_indexed.printSchema()

In [ ]:
def add_inverse_freq_weight(df: DataFrame, category_col: str) -> DataFrame:
    # count per category
    counts = df.groupBy(category_col).count()
    # total number of records
    total = df.count()
    # compute inverse frequency weight = total / count
    weights = counts.withColumn("weight", F.lit(total) / F.col("count")) \
                    .select(category_col, "weight")
    # join back to original DataFrame
    return df.join(weights, on=category_col, how="left")

# add weight column
df_weighted = add_inverse_freq_weight(df_indexed, "genre_index")


df_weighted.show(truncate=False)


In [ ]:
# 1. Add a random column
df_rand = df_weighted.withColumn("rand", F.rand(42))

# 2. Within each genre_index partition, order by rand and generate a row number
w_order = Window.partitionBy("genre_index").orderBy("rand")
df_rn = df_rand.withColumn("rn", F.row_number().over(w_order))

# 3. Within each genre_index partition, count the total number of rows
w_count = Window.partitionBy("genre_index")
df_cnt = df_rn.withColumn("cnt", F.count("*").over(w_count))

# 4. Compute the row-number threshold for the training set (floor(cnt * 0.8))
df_thresh = df_cnt.withColumn(
    "threshold",
    (F.col("cnt") * F.lit(0.8)).cast("integer")
).cache()

# 5. Split into training and testing sets based on row number and threshold
train_df = df_thresh.filter(F.col("rn") <= F.col("threshold")) \
    .drop("rand", "rn", "cnt", "threshold")
test_df  = df_thresh.filter(F.col("rn") >  F.col("threshold")) \
    .drop("rand", "rn", "cnt", "threshold")


In [ ]:
# Identify numeric feature columns excluding raw genre and binary flag
feature_cols = [
    c for c in train_df.columns
    if c not in ("track_id", "genre", "genre_index", "weight")
]

# Assemble features into a single vector column
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Apply assembler and drop the raw genre and binary flag columns
train_prepared = (
    assembler
      .transform(train_df)
      .drop("genre")
)

test_prepared = (
    assembler
      .transform(test_df)
      .drop("genre")
)

In [ ]:
show_as_html(train_prepared)

In [ ]:
# 3. Configure and train weighted multi‐class Logistic Regression
lr = LogisticRegression(
    featuresCol="features",   
    labelCol="genre_index", 
    standardization=True,
    weightCol="weight" # inverse‐frequency weight
)
lr_model = lr.fit(train_prepared)

# 4. Apply model on test set and cache predictions
predictions = lr_model.transform(test_prepared).cache()

# 5. Show class balance on the test set (before prediction)
show_class_balance(test_prepared, name="Test Set", labelCol="genre_index")

# # 6. Evaluate and print metrics

# 6.1 Multi‐class metrics: Accuracy, weighted Precision, weighted Recall
evaluator_acc    = MulticlassClassificationEvaluator(
    labelCol="genre_index",
    predictionCol="prediction",
    metricName="accuracy"
)
evaluator_prec   = MulticlassClassificationEvaluator(
    labelCol="genre_index",
    predictionCol="prediction",
    metricName="weightedPrecision"
)
evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="genre_index",
    predictionCol="prediction",
    metricName="weightedRecall"
)
evaluator_f1     = MulticlassClassificationEvaluator(
    labelCol="genre_index",
    predictionCol="prediction",
    metricName="f1"
)

accuracy  = evaluator_acc.evaluate(predictions)
precision = evaluator_prec.evaluate(predictions)
recall    = evaluator_recall.evaluate(predictions)
f1_score  = evaluator_f1.evaluate(predictions)

# 6.3 Print all metrics (replacing Average AUROC with F1)
print(f"Accuracy  = {accuracy:.4f}")
print(f"Precision = {precision:.4f}")
print(f"Recall    = {recall:.4f}")
print(f"F1 Score  = {f1_score:.4f}")


In [ ]:
confusion_matrix = (
    predictions
      .groupBy("genre_index")
      .pivot("prediction")
      .count()
      .orderBy("genre_index")
)

confusion_matrix.show()

In [ ]:
# convert to Pandas
cm_df = predictions.select("genre_index", "prediction").toPandas()


import pandas as pd
confusion_matrix = pd.crosstab(cm_df["genre_index"], cm_df["prediction"],
                               rownames=["Actual"], colnames=["Predicted"],
                               dropna=False)

# print(confusion_matrix)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,8))
sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
index_model

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

#  Extract true labels and predicted labels into a Pandas DataFrame
pred_df = predictions.select("genre_index", "prediction").toPandas()

# Convert the genre_index column to integer so that classification_report labels are "0", "1", …
pred_df["genre_index"] = pred_df["genre_index"].astype(int)

#  Generate a detailed classification report using scikit-learn
report = classification_report(
    pred_df["genre_index"],
    pred_df["prediction"],
    output_dict=True,
    zero_division=0  # Prevent division-by-zero warnings
)

# Step 3: Convert the report to a Pandas DataFrame and transpose it
metrics_df = pd.DataFrame(report).transpose()

# Step 4: Keep only the rows corresponding to each integer label (as strings)
metrics_by_class = metrics_df.loc[[str(i) for i in sorted(pred_df["genre_index"].unique())]]

# --- Begin modification: replace numeric index with original genre names ---

# Retrieve the list of genre names from the StringIndexer model
# index_model.labels is an array where the element at position i is the genre mapped to index i
genre_labels = index_model.labels  # e.g., ["Pop", "Rock", "Jazz", ...]

# Create a mapping from string index to genre name
index_to_genre = {str(i): genre_labels[i] for i in range(len(genre_labels))}

# Add a new column "genre" by mapping the current index (string) to the genre name
metrics_by_class["genre"] = metrics_by_class.index.map(index_to_genre)

# Set the new "genre" column as the DataFrame’s index, then drop the old numeric string index
metrics_by_class = metrics_by_class.set_index("genre")

# --- End modification ---

# Import plotting libraries

import seaborn as sns

# Step 5: Plot a heatmap of precision, recall, and F1-score for each genre (now indexed by name)
plt.figure(figsize=(12, 6))
sns.heatmap(
    metrics_by_class[["precision", "recall", "f1-score"]],
    annot=True,
    fmt=".2f",
    cmap="Blues"
)
plt.title("Per-Genre Precision, Recall, and F1-score")
plt.xlabel("Metric")
plt.ylabel("Genre")  # Now showing genre names on the y-axis
plt.tight_layout()
plt.show()

# Step 6: Print the metrics table with genre names as the index
print(metrics_by_class.round(3))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Suppose metrics_by_class is already indexed by genre name and contains the columns 'precision', 'recall', and 'f1-score'.

#  Prepare data for plotting
# Extract the three metric columns as a NumPy array, with each row corresponding to a genre
metric_values = metrics_by_class[["precision", "recall", "f1-score"]].values

# Number of genres
num_genres = metric_values.shape[0]

# The x locations for each genre on the x-axis
# We'll create an integer position for each genre: 0, 1, 2, ...
indices = np.arange(num_genres)

# Width of each bar within a group
bar_width = 0.25

#  Create the plot
plt.figure(figsize=(14, 6))

# Plot precision bars
plt.bar(
    indices,                            # x positions
    metric_values[:, 0],                # precision values
    bar_width,                          # width of each bar
    label="Precision",                  # legend label
    color="#4C72B0"                     # bar color (optional)
)

# Plot recall bars (shifted by bar_width to the right)
plt.bar(
    indices + bar_width,                # x positions + offset
    metric_values[:, 1],                # recall values
    bar_width,
    label="Recall",
    color="#55A868"
)

# Plot f1-score bars (shifted by 2 * bar_width)
plt.bar(
    indices + 2 * bar_width,            # x positions + 2*offset
    metric_values[:, 2],                # f1-score values
    bar_width,
    label="F1-Score",
    color="#C44E52"
)

#  Set x-axis ticks and labels
plt.xlabel("Genre", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("Per-Genre Precision, Recall, and F1-Score", fontsize=14)

# Place x-axis ticks at the centre of each group and label them with genre names
plt.xticks(
    indices + bar_width,                # Shift ticks to the right by half the group width to centre them
    metrics_by_class.index,             # List of genre names
    rotation=45,                        # Rotate labels to avoid overlap
    ha="right"                          # Align labels to the right
)

# Display the legend
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# stop_spark()